# All-in-One Multimodal Pipeline (Vision Transformer + Tabular Transformer)

This notebook contains an upgraded, state-of-the-art architecture for the thesis utilizing pure Self-Attention mechanisms:
1. **Vision Transformer (ViT)** for Candlestick Image Feature Extraction.
2. **Transformer Encoder** for Tabular Time-Series Sequence embedding.
3. **Cross-Attention** fusion.


In [ ]:
import pandas as pd
import numpy as np
import os
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, accuracy_score
from tqdm.notebook import tqdm
import copy
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Configs
CSV_PATH = 'btc_data_imputed.csv'
IMAGE_DIR = 'Candlestick'
OUTPUT_DIR = 'processed_data'
BATCH_SIZE = 32
SEQ_LENGTH = 12  # 12 hours total window
IMAGE_SIZE = (224, 224) # Standard ViT size requirements
EPOCHS = 20

### 1. Data Processing & Dataset Initialization

In [ ]:
class BTCMultimodalDataset(Dataset):
    def __init__(self, df, image_dir, seq_length=12, transform=None, mode='train'):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.seq_length = seq_length
        self.transform = transform
        self.mode = mode
        self.feature_cols = ['open', 'high', 'low', 'close', 'volume', 'sentiment_score']
        
    def __len__(self):
        return len(self.df) - self.seq_length
    
    def __getitem__(self, idx):
        tabular_seq = self.df.iloc[idx : idx + self.seq_length][self.feature_cols].values
        
        last_row = self.df.iloc[idx + self.seq_length - 1]
        img_name = last_row['index']
        year = str(img_name)[:4]
        img_path = os.path.join(self.image_dir, year, f"{img_name}.png")
        
        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            image = Image.new('RGB', (224,224), (0,0,0))
            
        if self.transform:
            image = self.transform(image)
            
        target_row = self.df.iloc[idx + self.seq_length]
        label = 1.0 if target_row['trend'] == 'Bullish' else 0.0
        
        tabular_seq = torch.tensor(tabular_seq, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.float32)
        
        return tabular_seq, image, label


In [ ]:
def prepare_data(csv_path, output_dir='processed_data'):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    df = pd.read_csv(csv_path)
    df['trend'] = df['trend'].fillna('Bearish') 
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values(by='timestamp')
    
    if os.path.exists('missing_images.csv'):
        missing_df = pd.read_csv('missing_images.csv')
        missing_indices = missing_df['index'].tolist()
        df = df[~df['index'].isin(missing_indices)].reset_index(drop=True)
        print(f"Dropped {len(missing_indices)} missing image rows from training sequence.")
    
    n = len(df)
    train_size = int(n * 0.70)
    val_size = int(n * 0.15)
    
    train_df = df.iloc[:train_size].copy()
    val_df = df.iloc[train_size : train_size+val_size].copy()
    test_df = df.iloc[train_size+val_size:].copy()
    
    scaler = RobustScaler()
    cols_to_scale = ['open', 'high', 'low', 'close', 'volume', 'sentiment_score']
    
    train_df[cols_to_scale] = scaler.fit_transform(train_df[cols_to_scale])
    val_df[cols_to_scale] = scaler.transform(val_df[cols_to_scale])
    test_df[cols_to_scale] = scaler.transform(test_df[cols_to_scale])
    
    train_df.to_csv(os.path.join(output_dir, 'train_btc_data.csv'), index=False)
    val_df.to_csv(os.path.join(output_dir, 'val_btc_data.csv'), index=False)
    test_df.to_csv(os.path.join(output_dir, 'test_btc_data.csv'), index=False)
    
    return train_df, val_df, test_df, scaler

train_df, val_df, test_df, scaler = prepare_data(CSV_PATH, OUTPUT_DIR)

# ViT typically uses slightly different ImageNet normalizations and scaling than ResNets,
# but the standard ImageNet values (0.485, 0.456, 0.406) work perfectly for the PyTorch ViT_B_16 implementation.
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = BTCMultimodalDataset(train_df, IMAGE_DIR, transform=transform, mode='train')
val_dataset = BTCMultimodalDataset(val_df, IMAGE_DIR, transform=transform, mode='val')
test_dataset = BTCMultimodalDataset(test_df, IMAGE_DIR, transform=transform, mode='test')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

### 2. Pure Transformer Architecture (SOTA)

In [ ]:
class PureTabularTransformer(nn.Module):
    def __init__(self, input_dim=6, seq_length=12, embed_dim=128, num_heads=4, num_layers=2):
        super(PureTabularTransformer, self).__init__()
        # Transform 6 features to internal embedding dim
        self.feature_embed = nn.Linear(input_dim, embed_dim)
        
        # Since Attention is invariant to order, we MUST add positional encodings for time-series
        self.pos_encoder = nn.Parameter(torch.randn(1, seq_length, embed_dim))
        
        encoder_layers = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, 
                                                    dim_feedforward=512, dropout=0.3, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        
        self.output_norm = nn.LayerNorm(embed_dim)
        
    def forward(self, x):
        # x shape: (B, Seq_length, Features)
        x = self.feature_embed(x) # (B, Seq, Embed_Dim)
        x = x + self.pos_encoder  # Add positional awareness
        
        # Pass through multi-head self attention blocks
        x = self.transformer(x)   # (B, Seq, Embed_Dim)
        
        # Global Average Pooling to condense the sequence into one representation
        x = x.mean(dim=1)         # (B, Embed_Dim)
        x = self.output_norm(x)
        
        return x

class ViTVisualEncoder(nn.Module):
    def __init__(self, out_dim=256):
        super(ViTVisualEncoder, self).__init__()
        # SOTA Vision Transformer (Base size, patch size 16)
        self.vit = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        
        # Replace the final classification head (for ImageNet's 1000 classes) with our embedding dimension
        num_ftrs = self.vit.heads.head.in_features
        self.vit.heads.head = nn.Sequential(
            nn.Linear(num_ftrs, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, out_dim)
        )
        
    def forward(self, x):
        return self.vit(x)

class MultimodalTransformerFusion(nn.Module):
    def __init__(self, tabular_input_dim=6):
        super(MultimodalTransformerFusion, self).__init__()
        self.tabular_encoder = PureTabularTransformer(input_dim=tabular_input_dim, embed_dim=128)
        self.visual_encoder = ViTVisualEncoder(out_dim=256)
        
        # To cross attend, dimensions must match. So we project Tabular (128) up to 256
        self.tabular_proj = nn.Linear(128, 256)
        
        self.cross_attention = nn.MultiheadAttention(embed_dim=256, num_heads=4, batch_first=True)
        
        # Classifier Output Shape = Tabular (256) + Visual Context (256) 
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),  # GELU is standard for Transformer architectures
            nn.Dropout(0.4),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1) # Probability logic
        )
        
    def forward(self, tabular_x, visual_x):
        tab_embed = self.tabular_encoder(tabular_x) # (B, 128)
        tab_embed = self.tabular_proj(tab_embed)    # Now (B, 256)
        
        vis_embed = self.visual_encoder(visual_x)   # (B, 256)
        
        q = vis_embed.unsqueeze(1)
        k = tab_embed.unsqueeze(1)
        v = tab_embed.unsqueeze(1)
        
        # Visual acts as query across the Tabular sequence embedding
        attn_output, _ = self.cross_attention(q, k, v)
        attn_output = attn_output.squeeze(1)
        
        fused_vector = torch.cat((attn_output, vis_embed), dim=1) # (B, 512)
        output = self.classifier(fused_vector)
        return output 

### 3. Training and Evaluation Loop Setup

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = MultimodalTransformerFusion().to(device)
optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4) # Scaled down LR is generally safer for ViTs
criterion = nn.BCEWithLogitsLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    epoch_loss = 0
    pbar = tqdm(dataloader, desc='Training')
    for tab_seq, img, label in pbar:
        tab_seq, img, label = tab_seq.to(device), img.to(device), label.to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        output = model(tab_seq, img)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    return epoch_loss / len(dataloader)

In [ ]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    epoch_loss = 0
    all_labels = []
    all_preds_probs = []
    
    with torch.no_grad():
        for tab_seq, img, label in tqdm(dataloader, desc='Evaluating'):
            tab_seq, img, label = tab_seq.to(device), img.to(device), label.to(device).unsqueeze(1)
            
            output = model(tab_seq, img)
            loss = criterion(output, label)
            epoch_loss += loss.item()
            
            probs = torch.sigmoid(output)
            all_labels.extend(label.cpu().numpy())
            all_preds_probs.extend(probs.cpu().numpy())
            
    all_labels = np.array(all_labels)
    all_preds_probs = np.array(all_preds_probs)
    all_preds_classes = (all_preds_probs >= 0.5).astype(int)
    
    metrics = {
        'loss': epoch_loss / len(dataloader),
        'accuracy': accuracy_score(all_labels, all_preds_classes),
        'roc_auc': roc_auc_score(all_labels, all_preds_probs) if len(np.unique(all_labels)) > 1 else 0,
        'f1': f1_score(all_labels, all_preds_classes, zero_division=0),
        'precision': precision_score(all_labels, all_preds_classes, zero_division=0),
        'recall': recall_score(all_labels, all_preds_classes, zero_division=0)
    }
    return metrics

In [ ]:
best_val_auc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

# Note: I have left this commented so it doesn't execute infinitely by default.
# You can just run the cell when ready!
"""
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 10)
    
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics = evaluate(model, val_loader, criterion, device)
    
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_metrics['loss']:.4f} | Val AUC: {val_metrics['roc_auc']:.4f} | Val F1: {val_metrics['f1']:.4f}")

    scheduler.step(val_metrics['roc_auc'])
    
    if val_metrics['roc_auc'] > best_val_auc:
        best_val_auc = val_metrics['roc_auc']
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(best_model_wts, 'best_transformer_model.pth')
        print("*** New Best Model Saved! ***")
        
print(f"\nTraining Complete. Best Validation ROC-AUC: {best_val_auc:.4f}")
"""

In [ ]:
"""
model.load_state_dict(torch.load('best_transformer_model.pth'))
test_metrics = evaluate(model, test_loader, criterion, device)
print("\n--- FINAL TEST METRICS ---")
print(f"Accuracy  : {test_metrics['accuracy']:.4f}")
print(f"ROC-AUC   : {test_metrics['roc_auc']:.4f}")
print(f"F1 Score  : {test_metrics['f1']:.4f}")
print(f"Precision : {test_metrics['precision']:.4f}")
print(f"Recall    : {test_metrics['recall']:.4f}")
"""